In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)


def generate_data(n_samples: int):
    ages = np.random.uniform(13, 100, n_samples)
    weights = np.random.uniform(40, 120, n_samples)

    # Synthetic rule: side effects depend on weight threshold
    side_effects = (weights >= 70).astype(int)

    X = np.column_stack((ages, weights))
    y = side_effects
    return X, y


def build_model(hidden_layers, learning_rate, loss_name):
    model = Sequential()
    model.add(Input(shape=(2,)))

    for units in hidden_layers:
        model.add(Dense(units, activation="relu"))

    model.add(Dense(2, activation="softmax"))

    if loss_name == "sparse_categorical_crossentropy":
        loss = "sparse_categorical_crossentropy"
    elif loss_name == "categorical_crossentropy":
        loss = "categorical_crossentropy"
    elif loss_name == "mse":
        loss = "mse"
    else:
        raise ValueError(f"Unsupported loss function: {loss_name}")

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=loss,
        metrics=["accuracy"]
    )
    return model


def prepare_targets(y_train, y_val, loss_name, num_classes=2):
    if loss_name in ["categorical_crossentropy", "mse"]:
        y_train_prepared = to_categorical(y_train, num_classes=num_classes)
        y_val_prepared = to_categorical(y_val, num_classes=num_classes)
    else:
        y_train_prepared = y_train
        y_val_prepared = y_val

    return y_train_prepared, y_val_prepared


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history["loss"], label="Train Loss")
    axes[0].plot(history.history["val_loss"], label="Validation Loss")
    axes[0].set_title(f"Loss - {title}")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="Train Accuracy")
    axes[1].plot(history.history["val_accuracy"], label="Validation Accuracy")
    axes[1].set_title(f"Accuracy - {title}")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def evaluate_model(model, X_test, y_test, scaler, title):
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(
        y_test,
        y_pred,
        target_names=["No Side Effects", "Side Effects"],
        output_dict=True
    )

    print(f"\n=== {title} ===")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["No Side Effects", "Side Effects"]
        )
    )

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No Side Effects", "Side Effects"],
        yticklabels=["No Side Effects", "Side Effects"]
    )
    plt.title(f"Confusion Matrix - {title}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    # Decision boundary
    h = 0.02
    x_min, x_max = X_test[:, 0].min() - 0.05, X_test[:, 0].max() + 0.05
    y_min, y_max = X_test[:, 1].min() - 0.05, X_test[:, 1].max() + 0.05
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    grid_scaled = np.c_[xx.ravel(), yy.ravel()]
    preds = model.predict(grid_scaled, verbose=0)
    Z = np.argmax(preds, axis=1).reshape(xx.shape)

    grid_unscaled = scaler.inverse_transform(grid_scaled)
    xx_unscaled = grid_unscaled[:, 0].reshape(xx.shape)
    yy_unscaled = grid_unscaled[:, 1].reshape(yy.shape)

    X_test_unscaled = scaler.inverse_transform(X_test)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx_unscaled, yy_unscaled, Z, alpha=0.35, cmap=plt.cm.RdBu)
    plt.scatter(
        X_test_unscaled[y_test == 0, 0],
        X_test_unscaled[y_test == 0, 1],
        c="blue",
        edgecolors="k",
        label="No Side Effects"
    )
    plt.scatter(
        X_test_unscaled[y_test == 1, 0],
        X_test_unscaled[y_test == 1, 1],
        c="red",
        edgecolors="k",
        label="Side Effects"
    )
    plt.xlabel("Age")
    plt.ylabel("Weight")
    plt.title(f"Decision Boundary - {title}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return {
        "accuracy": report["accuracy"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "confusion_matrix": cm,
        "report": report
    }


# Experimental setup
hidden_layer_configs = [[16, 32], [16, 32, 64]]
data_sizes = [1500, 3000]
learning_rates = [0.01, 0.001, 0.0001]
loss_functions = ["sparse_categorical_crossentropy", "categorical_crossentropy", "mse"]
epochs_options = [20, 30]
batch_sizes = [10, 50]

results = []

# Fixed independent test set
X_test_raw, y_test = generate_data(500)

experiment_id = 1
total_experiments = (
    len(hidden_layer_configs)
    * len(data_sizes)
    * len(learning_rates)
    * len(loss_functions)
    * len(epochs_options)
    * len(batch_sizes)
)

for hidden_layers in hidden_layer_configs:
    for data_size in data_sizes:
        X_raw, y = generate_data(data_size)

        X_train_raw, X_val_raw, y_train, y_val = train_test_split(
            X_raw, y, test_size=0.2, random_state=42
        )

        scaler = MinMaxScaler()
        X_train = scaler.fit_transform(X_train_raw)
        X_val = scaler.transform(X_val_raw)
        X_test = scaler.transform(X_test_raw)

        for learning_rate in learning_rates:
            for loss_name in loss_functions:
                y_train_prepared, y_val_prepared = prepare_targets(y_train, y_val, loss_name)

                for epochs in epochs_options:
                    for batch_size in batch_sizes:
                        config_name = (
                            f"Exp {experiment_id}/{total_experiments} | "
                            f"Layers={hidden_layers}, Data={data_size}, LR={learning_rate}, "
                            f"Loss={loss_name}, Epochs={epochs}, Batch={batch_size}"
                        )
                        print(config_name)

                        model = build_model(hidden_layers, learning_rate, loss_name)

                        history = model.fit(
                            X_train,
                            y_train_prepared,
                            validation_data=(X_val, y_val_prepared),
                            epochs=epochs,
                            batch_size=batch_size,
                            verbose=0
                        )

                        metrics = evaluate_model(model, X_test, y_test, scaler, config_name)
                        plot_history(history, config_name)

                        results.append({
                            "hidden_layers": str(hidden_layers),
                            "data_size": data_size,
                            "learning_rate": learning_rate,
                            "loss_function": loss_name,
                            "epochs": epochs,
                            "batch_size": batch_size,
                            "test_accuracy": metrics["accuracy"],
                            "weighted_f1": metrics["weighted_f1"],
                            "training_loss": history.history["loss"][-1],
                            "validation_loss": history.history["val_loss"][-1],
                            "training_accuracy": history.history["accuracy"][-1],
                            "validation_accuracy": history.history["val_accuracy"][-1]
                        })

                        experiment_id += 1

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by=["validation_loss", "weighted_f1"],
    ascending=[True, False]
).reset_index(drop=True)

print("\nTop configurations:")
print(results_df.head(10))

plt.figure(figsize=(14, 6))
sns.barplot(data=results_df.head(10), x="validation_loss", y=results_df.head(10).index.astype(str))
plt.title("Top 10 Configurations by Validation Loss")
plt.xlabel("Validation Loss")
plt.ylabel("Experiment Rank")
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
sns.barplot(data=results_df.head(10), x="weighted_f1", y=results_df.head(10).index.astype(str))
plt.title("Top 10 Configurations by Weighted F1")
plt.xlabel("Weighted F1")
plt.ylabel("Experiment Rank")
plt.tight_layout()
plt.show()

best_config = results_df.iloc[0]
print("\nBest configuration:")
print(best_config)

best_hidden_layers = eval(best_config["hidden_layers"])
best_data_size = int(best_config["data_size"])
best_learning_rate = float(best_config["learning_rate"])
best_loss_function = best_config["loss_function"]
best_epochs = int(best_config["epochs"])
best_batch_size = int(best_config["batch_size"])

# Re-train best model
X_best_raw, y_best = generate_data(best_data_size)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_best_raw, y_best, test_size=0.2, random_state=42
)

best_scaler = MinMaxScaler()
X_train_best = best_scaler.fit_transform(X_train_raw)
X_val_best = best_scaler.transform(X_val_raw)

y_train_best_prepared, y_val_best_prepared = prepare_targets(
    y_train, y_val, best_loss_function
)

best_model = build_model(best_hidden_layers, best_learning_rate, best_loss_function)

best_history = best_model.fit(
    X_train_best,
    y_train_best_prepared,
    validation_data=(X_val_best, y_val_best_prepared),
    epochs=best_epochs,
    batch_size=best_batch_size,
    verbose=0
)

print("\nBest model summary:")
best_model.summary()

plot_history(best_history, "Best Model")

# Save best model
model_path = "../models/best_side_effects_model.keras"
best_model.save(model_path)
print(f"Saved model to: {model_path}")

# Load and test saved model on new unseen data
loaded_model = load_model(model_path)

X_new_raw, y_new = generate_data(500)
X_new = best_scaler.transform(X_new_raw)

y_new_prob = loaded_model.predict(X_new, verbose=0)
y_new_pred = np.argmax(y_new_prob, axis=1)

print("\nEvaluation on new unseen data:")
print(classification_report(y_new, y_new_pred, target_names=["No Side Effects", "Side Effects"]))

comparison_df = pd.DataFrame({
    "Age": X_new_raw[:, 0],
    "Weight": X_new_raw[:, 1],
    "Actual": y_new,
    "Predicted": y_new_pred
})

print("\nSample predictions:")
print(comparison_df.head(10))